# Bank Complaint Router

Fine-tune Qwen3-1.7B-Base on the approved January–February 2026 CFPB training split. The model learns to return one of seven routing labels.

1. In Colab, select **Runtime → Change runtime type → T4 GPU**.
2. Run the cells in order. When prompted, upload the unmodified `Week5/data/processed/train.csv` (3,360 rows). Keep `val.csv` for Phase 3.
3. Download the export bundle from the Colab Files pane before the runtime disconnects.

The [LLaMA-Factory Qwen3 example](https://github.com/hiyouga/LlamaFactory/blob/v0.9.4/examples/train_lora/qwen3_lora_sft.yaml) uses the `qwen3_nothink` template. This notebook uses the same template for training and export so the target is a single label.

Before running the numbered sections, connect to a T4 GPU and run `!nvidia-smi -L` in a temporary code cell. Download the final model ZIP before ending the session. See the Week5 COLAB.md runbook for the file checklist and download steps.

## 1. Install LLaMA-Factory and set up the run

In [ ]:
import csv
import hashlib
import json
import subprocess
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "git+https://github.com/hiyouga/LlamaFactory.git@v0.9.4",
    "matplotlib",
], check=True)

In [ ]:
import yaml
from google.colab import files

MODEL_ID = "Qwen/Qwen3-1.7B-Base"
TEMPLATE = "qwen3_nothink"
LABELS = (
    "Credit reporting",
    "Debt collection",
    "Credit card",
    "Checking or savings account",
    "Mortgage",
    "Student loan",
    "Money transfer",
)
SYSTEM_PROMPT = "You are a bank complaint router. Reply with exactly one label from: " + ", ".join(LABELS) + "."
TRAIN_SHA256 = "2d200e6e0b17dc0c0dd7f9cbc682b6be535631a560f59b6edbb45d30874ffd6f"

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
RUN_DIR = Path("/content/complaint_router") / RUN_ID
DATASET_DIR = RUN_DIR / "dataset"
ADAPTER_DIR = RUN_DIR / "adapter"
MERGED_DIR = RUN_DIR / "merged_model"
RESULTS_DIR = RUN_DIR / "eval_results"
DATASET_DIR.mkdir(parents=True)
RESULTS_DIR.mkdir()
print(f"Run directory: {RUN_DIR}")

## 2. Convert the approved training split to ShareGPT

Each record contains the routing prompt, the complaint narrative, and the expected label. The upload checksum ties this run to the exact Phase 1 split.

In [ ]:
import io

uploaded = files.upload()
if set(uploaded) != {"train.csv"}:
    raise ValueError("Upload only Week5/data/processed/train.csv.")
train_bytes = uploaded["train.csv"]
if hashlib.sha256(train_bytes).hexdigest() != TRAIN_SHA256:
    raise ValueError("The CSV differs from the approved January–February training split.")

reader = csv.DictReader(io.StringIO(train_bytes.decode("utf-8-sig"), newline=""))
if reader.fieldnames != ["text", "label"]:
    raise ValueError("Expected the columns text,label.")
rows = list(reader)
counts = Counter(row["label"] for row in rows)
if counts != Counter({label: 480 for label in LABELS}) or any(not row["text"].strip() for row in rows):
    raise ValueError("Expected 480 populated narratives for each of the seven labels.")

records = [
    {
        "system": SYSTEM_PROMPT,
        "conversations": [
            {"from": "human", "value": row["text"]},
            {"from": "gpt", "value": row["label"]},
        ],
    }
    for row in rows
]
(DATASET_DIR / "complaint_router.json").write_text(
    json.dumps(records, ensure_ascii=False), encoding="utf-8"
)
dataset_info = {
    "complaint_router": {
        "file_name": "complaint_router.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "system": "system"},
    }
}
(DATASET_DIR / "dataset_info.json").write_text(
    json.dumps(dataset_info, indent=2), encoding="utf-8"
)
print(f"Registered {len(records):,} training examples. Counts: {dict(counts)}")

## 3. Train the rank-8 adapter

Start with FP16 mixed precision, a microbatch of 2, and 8 accumulation steps for an effective batch of 16 on one GPU. If training runs out of GPU memory, restart from setup with a microbatch of 1 and 16 accumulation steps.

In [ ]:
train_config = {
    "model_name_or_path": MODEL_ID,
    "template": TEMPLATE,
    "dataset_dir": str(DATASET_DIR),
    "dataset": "complaint_router",
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_rank": 8,
    "lora_target": "all",
    "cutoff_len": 512,
    "output_dir": str(ADAPTER_DIR),
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "num_train_epochs": 2.0,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "fp16": True,
    "seed": 42,
    "data_seed": 42,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "report_to": "none",
}
train_path = RUN_DIR / "train.yaml"
train_path.write_text(yaml.safe_dump(train_config, sort_keys=False), encoding="utf-8")
subprocess.run(["llamafactory-cli", "train", str(train_path)], check=True)

## 4. Save the training loss curve

In [ ]:
import matplotlib.pyplot as plt

state = json.loads((ADAPTER_DIR / "trainer_state.json").read_text(encoding="utf-8"))
loss_history = [
    {"step": entry["step"], "loss": entry["loss"]}
    for entry in state["log_history"]
    if "loss" in entry
]
if not loss_history:
    raise RuntimeError("No step losses found in trainer_state.json.")
(RESULTS_DIR / "training_loss.json").write_text(
    json.dumps(loss_history, indent=2), encoding="utf-8"
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([entry["step"] for entry in loss_history], [entry["loss"] for entry in loss_history])
ax.set(xlabel="Optimizer step", ylabel="Training loss", title="Complaint router LoRA training")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "loss_curve.png", dpi=160)
plt.show()
plt.close(fig)

## 5. Merge and export the model

The [LLaMA-Factory export command](https://github.com/hiyouga/LlamaFactory/blob/v0.9.4/examples/merge_lora/qwen3_lora_sft.yaml) merges the adapter into the unquantized base weights and saves the model and tokenizer in the Colab runtime.

In [ ]:
merge_config = {
    "model_name_or_path": MODEL_ID,
    "adapter_name_or_path": str(ADAPTER_DIR),
    "template": TEMPLATE,
    "finetuning_type": "lora",
    "infer_dtype": "float16",
    "export_dir": str(MERGED_DIR),
    "export_size": 2,
    "export_device": "cpu",
    "export_legacy_format": False,
}
merge_path = RUN_DIR / "merge.yaml"
merge_path.write_text(yaml.safe_dump(merge_config, sort_keys=False), encoding="utf-8")
subprocess.run(["llamafactory-cli", "export", str(merge_path)], check=True)

## 6. Smoke-test the merged model

Run one hand-written complaint per queue with greedy decoding. Review the exact-match result for each example. Held-out metrics come from Phase 3.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

samples = {
    "Credit reporting": "My credit report shows a 60-day late payment on an account I paid on time. The bureau refused to correct it.",
    "Debt collection": "A collection agency calls my workplace every day and refuses to send written validation of the debt it says I owe.",
    "Credit card": "My credit card issuer charged interest after I paid my statement balance in full before the due date.",
    "Checking or savings account": "My bank placed a hold on my paycheck deposit in my checking account and charged me an overdraft fee.",
    "Mortgage": "My mortgage servicer stopped processing my loan modification and scheduled a foreclosure sale on my home.",
    "Student loan": "My student loan servicer counted none of my qualifying payments toward Public Service Loan Forgiveness.",
    "Money transfer": "Someone took over my Zelle profile and sent an unauthorized instant transfer. I need the transfer stopped.",
}

tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.float16).to("cuda").eval()
results = []
with torch.inference_mode():
    for expected, complaint in samples.items():
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": complaint},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)
        generated = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            stop_strings=list(LABELS),
            tokenizer=tokenizer,
            pad_token_id=tokenizer.pad_token_id,
        )
        completion = generated[0, inputs.input_ids.shape[1]:]
        predicted = tokenizer.decode(completion, skip_special_tokens=True).strip()
        passed = predicted == expected
        results.append({"expected": expected, "predicted": predicted, "passed": passed})
        print(f"{'PASS' if passed else 'FAIL'} | {expected} | {predicted!r}")

(RESULTS_DIR / "smoke_tests.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Exact matches: {sum(result['passed'] for result in results)}/{len(samples)}")
del model
torch.cuda.empty_cache()

## 7. Bundle the export

The bundle contains the merged Transformers model, loss curve, training loss, smoke-test results, and run configuration. Refresh the Colab Files pane and download the ZIP at the path printed below. Extract it locally under `Week5` when ready for Phase 3.

In [ ]:
from zipfile import ZIP_STORED, ZipFile

manifest = {
    "base_model": MODEL_ID,
    "llamafactory_version": "v0.9.4",
    "dataset_period": "2026-01 through 2026-02",
    "train_csv_sha256": TRAIN_SHA256,
    "training_rows": len(rows),
    "template": TEMPLATE,
    "labels": LABELS,
    "system_prompt": SYSTEM_PROMPT,
}
manifest_path = RUN_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
bundle_path = Path("/content") / f"complaint-router-{RUN_ID}.zip"
with ZipFile(bundle_path, "w", compression=ZIP_STORED) as bundle:
    for path in (train_path, merge_path, manifest_path, *MERGED_DIR.rglob("*"), *RESULTS_DIR.iterdir()):
        if path.is_file():
            bundle.write(path, path.relative_to(RUN_DIR))
print(f"Download {bundle_path} ({bundle_path.stat().st_size / 1024**3:.2f} GiB).")